# Laboratorium 5: co mówi współczynnik i co pokazują reszty?

Jak porównać dwie grupy klientów? Skąd wiemy, że prosta wystarcza do opisu danych?
Jak ocenić niepewność wyniku, gdy rozrzut błędów nie jest stały?

[Karta laboratorium](https://rodakt.github.io/MAD/laby/05-regresja-liniowa-2.html)

Zapisz kopię notebooka na swoim Dysku (*Plik → Zapisz kopię na Dysku*).
Kod wpisuj samodzielnie w pustych komórkach; podpowiedzi otwieraj, gdy ich potrzebujesz.
Przed obliczeniem spróbuj przewidzieć wynik. Po obliczeniu zapisz interpretację
w komórce „Twoja odpowiedź” (dwuklik pozwala ją edytować).

Części 1--2 zajmują około 35 minut, część 3 -- 25 minut, część 4 -- 25 minut.
Ostatnie 5 minut przeznaczamy na wnioski. Współliniowość na końcu jest opcjonalna.

## Przygotowanie

Uruchom gotową komórkę. Wczytuje dwa zbiory z rozdziału 6 książki:

- **Credit**: 400 klientów banku. `Balance` to średnie saldo zadłużenia na karcie
  w USD, `Income` -- dochód w tysiącach USD, `Student` -- `Yes` albo `No`.
- **Auto**: 392 samochody po usunięciu braków w `mpg` i `horsepower`.
  `horsepower` oznacza moc silnika w KM, a `mpg` -- liczbę mil na galon paliwa.
  Większe `mpg` oznacza większą oszczędność paliwa.

Najpierw pracujemy na `credit`, a od części 3 na `auto`.
Badamy interpretację i niepewność oszacowań na tych danych.
Nie oceniamy tu błędu na nowych danych; ten cel wymaga osobnej oceny jak na labie 4.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from statsmodels.nonparametric.smoothers_lowess import lowess

base_url = "https://rodakt.github.io/MAD/laby/dane/"
credit = pd.read_csv(base_url + "Credit.csv")
auto = pd.read_csv(base_url + "Auto.csv", na_values="?").dropna(
    subset=["mpg", "horsepower"]
).copy()

print(f"Credit: {len(credit)} klientów; Auto: {len(auto)} samochody.")

## 1. Cecha jakościowa: kogo z kim porównujemy?

### 1.1. Dwie średnie i jedna zmienna zero-jedynkowa

Oblicz średnie `Balance` osobno dla `Student = No` i `Student = Yes`.
Zapisz jako `mean_balance` i oblicz różnicę: studenci minus pozostali klienci.

Dodaj do `credit` kolumnę `student`, równą 1 dla studentów i 0 dla pozostałych osób.
Przyjmujemy model
$$\widehat y=\widehat\beta_0+\widehat\beta_S s.$$
Zanim go dopasujesz: jakie współczynniki powinien dać OLS, skoro model przewiduje
jedną wspólną wartość w każdej grupie?

Dopasuj `Balance ~ student` jako `model_group`. Odczytaj współczynniki
przez `.params` i sprawdź swoje przewidywanie. Zinterpretuj je w dolarach.
Czy różnica oznacza, że zostanie studentem powoduje zmianę zadłużenia?

<details>
<summary>Podpowiedź 1: grupy i kodowanie</summary>

`credit.groupby("Student")["Balance"].mean()` oblicza średnie w grupach.
`(credit["Student"] == "Yes").astype(int)` zamienia porównanie na zera i jedynki.

</details>

<details>
<summary>Podpowiedź 2: dopasowanie</summary>

Jak na labie 4: `smf.ols("Balance ~ student", data=credit).fit()`.
Wyraz wolny biblioteka nazywa `Intercept`, a współczynnik grupy -- `student`.

</details>

**Twoja odpowiedź:**

### 1.2. Dodaj dochód: czy porównanie nadal jest takie samo?

Dopasuj `model_add`, uwzględniający dochód oraz zmienną `student`.
Przed obliczeniem przewidź: czy współczynnik grupy musi nadal być równy różnicy
średnich z 1.1? Czy znasz z góry kierunek jego zmiany?

Zbuduj tabelę `clients` z czterema klientami w podanej kolejności:

| Wiersz | Income | student |
|---|---:|---:|
| 0 | 30 | 0 |
| 1 | 30 | 1 |
| 2 | 60 | 0 |
| 3 | 60 | 1 |

Oblicz `pred_add` i różnice predykcji 1−0 oraz 3−2.
Porównaj je ze współczynnikiem `student`. Co jest teraz ustalone w porównaniu grup?
Czy model pozwala, aby różnica między grupami zależała od dochodu?

<details>
<summary>Podpowiedź: formuła i nowi klienci</summary>

Postępuj jak z `model_add` i tabelą `markets` w części 2.2 labu 4:
cechy w formule łączymy znakiem `+`, a tabelę budujemy przez `pd.DataFrame`
ze słownika kolumn. Predykcje odczytujemy przez `.iloc`.

</details>

**Twoja odpowiedź:**

## 2. Interakcja: poziom czy także nachylenie?

### 2.1. Dwie proste dla dwóch grup

Dodaj do `credit` kolumnę `Income_student` jako iloczyn dochodu i zmiennej `student`,
tak jak kolumnę `TV_radio` w części 4.1 labu 4. Ile wynosi ona dla osób niebędących studentami?
Dopasuj `model_int` zawierający `Income`, `student` oraz `Income_student`.
Zachowaj oba efekty główne.

Odczytaj współczynniki. Dla każdej grupy zapisz osobno równanie przewidywanego salda:
$$
\begin{aligned}
s=0:\quad &\widehat y=\widehat\beta_0+\widehat\beta_I x,\\
s=1:\quad &\widehat y=(\widehat\beta_0+\widehat\beta_S)
                    +(\widehat\beta_I+\widehat\beta_{IS})x.
\end{aligned}
$$
Wstaw otrzymane liczby. Co oznacza współczynnik interakcji w USD na dodatkowy
1000 USD dochodu? Co oznacza teraz sam współczynnik `student`?

Przed obliczeniem nowych predykcji przewidź, czy różnica między grupami
będzie większa przy dochodzie 30 czy 60 tys. USD. Dodaj kolumnę iloczynu do `clients`
i sprawdź. Dla obu modeli narysuj po dwie proste względem dochodu, dla `student = 0` i `student = 1`,
w zakresie 10--100 tys. USD. Zrób dwa panele albo dwa osobne wykresy i opisz grupy.

<details>
<summary>Podpowiedź 1: interpretacja</summary>

Równania grup otrzymujemy tak jak nachylenie TV zależne od radia w labie 4
(„Co zmieniło się w równaniu?”): wstawiamy $s=0$ albo $s=1$ i grupujemy składniki.
Różnica „student minus pozostały klient” przy dochodzie $x$ wynosi
$\widehat\beta_S+\widehat\beta_{IS}x$.
Sam $\widehat\beta_S$ dotyczy $x=0$. Sprawdź, czy taki dochód występuje w danych.

</details>

<details>
<summary>Podpowiedź 2: wykres</summary>

`grid = np.linspace(10, 100, 100)` tworzy siatkę dochodów.
`pd.DataFrame({"Income": grid, "student": 0, "Income_student": 0})` tworzy klientów
pierwszej grupy. W drugiej tabeli `student = 1`, a iloczyn jest równy `grid`.
Predykcje rysujemy przez `plt.plot`.
`plt.legend()` pokazuje opisy przekazane jako `label`.

</details>

**Twoja odpowiedź:**

### 2.2. Czy dane przeczą równoległości?

Zapisz hipotezę o równoległych prostych jako hipotezę o jednym współczynniku.
Odczytaj jego $p$-wartość i sformułuj wniosek przy $\alpha=0{,}05$.
Czy brak odrzucenia hipotezy dowodzi identyczności nachyleń?
Czy brak interakcji oznacza brak różnicy między grupami?

Klasyczny test $t$ interpretujemy przy założeniach modelu liniowego z rozdziału 6;
rysunek dwóch prostych sam tych założeń nie sprawdza.

Ręcznie tworzona kolumna iloczynu pokazuje, że interakcja jest zwykłą cechą.
W praktyce formuła `Balance ~ Income + student + Income:student` daje ten sam model:
`:` dodaje iloczyn cech, a `predict` oblicza go sam w nowych danych.

<details>
<summary>Podpowiedź: wybrana p-wartość</summary>

Jak w części 4.3 labu 4: $p$-wartość współczynnika `Income_student` wybieramy z `.pvalues`.

</details>

**Twoja odpowiedź:**

## 3. Diagnostyka: co pozostało w resztach?

### 3.1. Prosta na danych Auto

Przechodzimy do `auto`. Dopasuj `model_lin`: `mpg ~ horsepower`.
Narysuj dane oraz prostą predykcji.

Oblicz reszty `resid_lin` tak jak błędy w części 1.2 labu 4: obserwacja minus predykcja.
Porównaj kilka pierwszych z gotowym polem `model_lin.resid`.
Następnie narysuj reszty względem predykcji, z poziomą linią na wysokości zera.
Dodaj wygładzenie LOWESS, żeby łatwiej odczytać kształt chmury reszt.

Opisz znak reszt na obu krańcach i w środku zakresu predykcji.
Gdzie model systematycznie przewiduje za mało, a gdzie za dużo?
Czy jest to problem z postacią średniej, czy tylko z rozrzutem błędów?

<details>
<summary>Podpowiedź 1: dane do wykresu reszt</summary>

`model_lin.fittedvalues` zawiera $\widehat y$ dla danych uczących.
`.head()` pokazuje pierwsze wartości kolumny.
`plt.axhline(0, color="black")` dodaje poziomą linię.

</details>

<details>
<summary>Podpowiedź 2: wygładzenie</summary>

`smooth = lowess(resid_lin, model_lin.fittedvalues, frac=0.4)` daje
pary współrzędnych uporządkowane wzdłuż osi poziomej.
Dodaj `plt.plot(smooth[:, 0], smooth[:, 1], color="red")` do wykresu reszt.
Wygładzenie pomaga odczytać wzorzec, nie jest osobnym testem założeń.

</details>

**Twoja odpowiedź:**

### 3.2. Dodaj kwadrat i ponownie obejrzyj reszty

Dodaj do `auto` kolumnę `horsepower_sq` jako kwadrat mocy, tak jak iloczyn
w części 2.1. Dopasuj `model_sq`, zawierający moc oraz jej kwadrat.
Przewidź, co powinno zmienić się na wykresie reszt, jeżeli kwadrat pomaga opisać średnią.

Powtórz wykres reszt z wygładzeniem, z takimi samymi zakresami osi jak w 3.1,
żeby łatwiej porównać oba modele. W razie potrzeby narysuj oba wykresy ponownie obok siebie.
Co stało się z łukiem? Czy szerokość pasa reszt jest podobna dla małych i dużych predykcji?

Oceń zdanie: „Dodałem kwadrat, więc klasyczne błędy standardowe i testy są już na pewno poprawne”.
Wyjaśnij też, dlaczego model z kwadratem nadal jest liniowy w parametrach.

<details>
<summary>Podpowiedź: wspólne osie</summary>

`fig, axes = plt.subplots(1, 2, sharex=True, sharey=True)` tworzy dwa panele
ze wspólnymi zakresami osi. Na każdym `ax` używamy `ax.scatter`, `ax.plot`,
`ax.axhline`, `ax.set_xlabel` i `ax.set_title`.

</details>

**Twoja odpowiedź:**

## 4. Bootstrap par: jak niepewna jest przewidywana średnia?

Zostajemy przy ustalonym modelu z kwadratem. Interesuje nas jedna liczba:
**przewidywana średnia liczba mil na galon dla samochodów o mocy 70 KM**.
Dla kolejnych prób klientów czy samochodów oszacowania byłyby inne.
Mamy jedną próbę, więc tę zmienność przybliżymy bootstrapem.

Losujemy ze zwracaniem całe wiersze $(x_i,y_i)$ i ponownie dopasowujemy model.
Taki bootstrap par zachowuje związek cech z rozrzutem wyniku i nie wymaga stałej
wariancji błędów. Zakładamy niezależne obserwacje z tego samego rozkładu.
Bootstrap nie naprawia błędnej postaci średniej ani zależności między obserwacjami.

### 4.1. Ustal wielkość, którą będziemy oszacowywać

Zbuduj `car_70` -- tabelę z jednym wierszem dla mocy 70 KM
oraz poprawnym kwadratem mocy. Oblicz `mean_70` jako pojedynczą
predykcję z dopasowanego już `model_sq`.

Czy ta liczba ma opisywać każdy samochód o mocy 70 KM, czy średnią według modelu?
Jakiego rozrzutu nie opisuje niepewność tej średniej?

<details>
<summary>Podpowiedź: jedna predykcja</summary>

Tabela do predykcji musi zawierać wszystkie cechy modelu, także obliczone:
tak jak iloczyn w `clients` i w `markets` z labu 4. Tutaj są to `horsepower`
i `horsepower_sq`. `predict` przyjmuje tabelę także wtedy, gdy zawiera tylko
jeden wiersz; pojedynczą wartość wybierz przez `.iloc[0]`.

</details>

**Twoja odpowiedź:**

### 4.2. Jedna próba bootstrapowa

Utwórz generator `rng_one = np.random.default_rng(2026)` i zapisz liczbę samochodów jako `n`.
Wylosuj ze zwracaniem `n` indeksów wierszy, a następnie wybierz je przez `.iloc` jako `sample`.

Przed dopasowaniem sprawdź: ile jest wszystkich wierszy, a ile różnych indeksów?
Czy próba zawiera samochody wielokrotnie? Dlaczego samo przetasowanie wierszy
nie wystarczyłoby do oceny niepewności OLS?

Dopasuj na `sample` ten sam model z kwadratem i oblicz predykcję dla `car_70`.
Porównaj ją z `mean_70`. Czy jedna replikacja wystarcza do oceny rozrzutu oszacowania?

<details>
<summary>Podpowiedź 1: losowanie wierszy</summary>

`rng_one.integers(0, n, size=n)` losuje indeksy ze zwracaniem.
`auto.iloc[idx]` wybiera całe wiersze -- wynik wraz ze wszystkimi cechami.
`np.unique(idx).size` liczy różne wylosowane indeksy.

</details>

<details>
<summary>Podpowiedź 2: ponowne dopasowanie</summary>

W wywołaniu `smf.ols` zachowaj formułę z 3.2, ale przekaż `data=sample`.
Dla nowego modelu ponownie wywołaj `predict(car_70)`.

</details>

**Twoja odpowiedź:**

### 4.3. Od jednej replikacji do rozkładu

Utwórz osobny generator `rng = np.random.default_rng(2026)`.
Napisz pętlę, która `B` razy wykonuje kroki z 4.2 i zbiera pojedyncze predykcje
dla `car_70` w liście `boot_mean`. **Model trzeba dopasować od nowa w każdej replikacji.**
Najpierw sprawdź kod dla `B = 20`, a po sprawdzeniu uruchom dla `B = 2000`;
przed każdym pełnym uruchomieniem odtwórz generator i pustą listę.

Narysuj histogram predykcji bootstrapowych i zaznacz pionową linią `mean_70`.
Oblicz:

1. błąd standardowy `se_boot` jako odchylenie standardowe replikacji,
2. przedział percentylowy `ci_boot` z kwantyli 2,5% i 97,5%.

Czy odchylenie standardowe replikacji trzeba jeszcze dzielić przez $\sqrt B$?
Czy większe `B` zastępuje większą liczbę samochodów w danych?
Czy ten przedział ma obejmować 95% wyników pojedynczych samochodów o mocy 70 KM?

<details>
<summary>Podpowiedź 1: plan pętli</summary>

Zacznij od `boot_mean = []`. W każdej iteracji wylosuj indeksy, wybierz wiersze,
dopasuj model i dopisz pojedynczą predykcję przez `boot_mean.append(value)`.
Po pętli zamień listę na tablicę: `boot_mean = np.array(boot_mean)`.
Nie losuj cech i wyników osobno -- zniszczyłoby to ich związek.

</details>

<details>
<summary>Podpowiedź 2: podsumowanie replikacji</summary>

`boot_mean.std(ddof=1)` oblicza odchylenie standardowe.
`np.quantile(boot_mean, [0.025, 0.975])` zwraca oba końce przedziału.
`plt.hist(boot_mean, bins=40)` rysuje histogram,
a `plt.axvline(mean_70, color="red")` dodaje linię.

</details>

**Twoja odpowiedź:**

### 4.4. Porównaj ze wzorem klasycznym

Dla tej samej wielkości odczytaj klasyczny błąd standardowy `se_classic`
i porównaj go z `se_boot`. Który jest większy? Jak to się wiąże z szerokością
pasa reszt w 3.2 przy dużych predykcjach mpg, czyli przy małej mocy?
Co zmieniło się w sposobie oceny niepewności, a co pozostało takie samo w modelu?

<details>
<summary>Podpowiedź: niepewność średniej w nowym punkcie</summary>

`model_sq.get_prediction(car_70).summary_frame()` daje tabelę:
`mean` to predykcja średniej, `mean_se` -- jej klasyczny błąd standardowy,
a `mean_ci_lower` i `mean_ci_upper` -- końce klasycznego przedziału dla średniej.
Kolumny `obs_ci_lower` i `obs_ci_upper` dotyczą **pojedynczej nowej obserwacji**.
Nie porównuj bootstrapu średniej z przedziałem dla obserwacji.

</details>

**Twoja odpowiedź:**

## 5. Wnioski

Bez nowych obliczeń odpowiedz:

1. Co porównuje współczynnik `student` w modelu z samą grupą,
   w modelu addytywnym z dochodem i w modelu z interakcją?
2. Jaki problem zmniejszyło dodanie kwadratu mocy? Jaki pozostał widoczny w resztach?
3. Czego dotyczy przedział bootstrapowy z części 4 i czego nie można z niego wywnioskować
   o pojedynczym samochodzie?
4. Co powinna zawierać replikacja bootstrapowa, żeby zachować związek mocy z wynikiem?

**Twoje odpowiedzi:**

## Opcjonalnie: dlaczego współczynnik nagle się zmienia?

Wróć do `credit`. Dopasuj modele `Balance ~ Limit` oraz `Balance ~ Limit + Rating`.
Przed porównaniem sprawdź korelację `Limit` i `Rating`.

Zestaw współczynnik `Limit` i jego błąd standardowy w obu modelach.
Następnie porównaj predykcje obu modeli na obserwowanych klientach:
oblicz pierwiastek ze średniego kwadratu różnic predykcji oraz porównaj go
z odchyleniem standardowym `Balance`.

Czy duża zmiana współczynnika oznacza równie dużą zmianę predykcji?
Dlaczego trudno przypisać osobne związki z saldem dwóm niemal identycznym informacjom?
Czy ta zgodność predykcji na obserwowanych klientach gwarantuje zgodność dla dowolnych
nowych kombinacji limitu i oceny?

<details>
<summary>Podpowiedź: korelacja i różnice predykcji</summary>

`credit["Limit"].corr(credit["Rating"])` oblicza korelację.
Po odjęciu dwóch tablic predykcji użyj `np.sqrt((difference**2).mean())`.
Punktem odniesienia jest `credit["Balance"].std()`.

</details>

**Twoja odpowiedź:**

## Fragmenty książki

- [Cechy jakościowe](https://rodakt.github.io/MAD_book/2-modele-liniowe/06-regresja-liniowa.html#sec-cechy-jakosciowe).
- [Interakcje i nieliniowość](https://rodakt.github.io/MAD_book/2-modele-liniowe/06-regresja-liniowa.html#sec-interakcje-nieliniowosc).
- [Diagnostyka modelu](https://rodakt.github.io/MAD_book/2-modele-liniowe/06-regresja-liniowa.html#sec-diagnostyka).
- [Bootstrap](https://rodakt.github.io/MAD_book/1-fundamenty/05-ocena-modeli.html#sec-bootstrap): algorytm, błąd standardowy i przedział percentylowy.